In [1]:
import pandas as pd
import pathlib
import numpy as np

In [2]:
BOX_DIR = pathlib.Path(r"E:\Box\Plan Bay Area 2050+\Performance and Equity\Plan Performance\Equity_Performance_Metrics\Final_Blueprint\metrics_input_files")
MODEL_DIR = pathlib.Path(r"M:\urban_modeling\baus\PBA50Plus\PBA50Plus_FinalBlueprint\PBA50Plus_Final_Blueprint_v65")

In [3]:
# SLR inundation to parcel lookup
slr_lookup = pd.read_csv(BOX_DIR / "urbansim_slr_MAR2025.csv")
print('rows of slr_lookup:', slr_lookup.shape[0], '; unique parcel_id:', slr_lookup['parcel_id'].nunique())
display(slr_lookup.head(3))

rows of slr_lookup: 6093 ; unique parcel_id: 6093


,parcel_id,inundation
0,1788166.0,20.0
1,1739836.0,10.0
2,1739905.0,10.0


In [4]:
# 2050 parcel output
parcels_2050 = pd.read_csv(MODEL_DIR / "core_summaries" / "PBA50Plus_Final_Blueprint_v65_parcel_summary_2050.csv")
print('rows of parcels_2050:', parcels_2050.shape[0], '; unique parcel_id:', parcels_2050['parcel_id'].nunique())
display(parcels_2050.head(3))

rows of parcels_2050: 1956212 ; unique parcel_id: 1956212


,parcel_id,geom_id,y,x,max_far,built_dua,max_dua,built_far,non_residential_sqft,residential_units,...,hhq3,hhq4,tothh,AGREMPN,FPSEMPN,HEREMPN,MWTEMPN,OTHEMPN,RETEMPN,totemp
0,229116,10305106092872,37.655379,-121.795620,0.00,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,244166,11107351665227,37.717277,-121.713004,0.35,0.000000,3.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,202378,11030175960628,37.655260,-122.014199,0.00,1.133806,0.0,0.116293,0.0,17.0,...,3.0,5.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# merge
parcels_withSLR_2050 = pd.merge(
    parcels_2050[['parcel_id', 'totemp', 'residential_units', 'non_residential_sqft', 'tothh', 'hhq1', 'hhq2', 'hhq3', 'hhq4']],
    slr_lookup,
    on='parcel_id',
    how='left'
)
print('rows of parcels_withSLR_2050:', parcels_withSLR_2050.shape[0], '; unique parcel_id:', parcels_withSLR_2050['parcel_id'].nunique())
display(parcels_withSLR_2050.head(3))

rows of parcels_withSLR_2050: 1956212 ; unique parcel_id: 1956212


,parcel_id,totemp,residential_units,non_residential_sqft,tothh,hhq1,hhq2,hhq3,hhq4,inundation
0,229116,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,244166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,202378,0.0,17.0,0.0,17.0,2.0,7.0,3.0,5.0,NaN


In [6]:
# summarize inundation impacts
parcels_withSLR_2050['inundation_tag'] = parcels_withSLR_2050['inundation'].notnull()
inundation_summary = parcels_withSLR_2050.groupby('inundation_tag').agg(
    totemp=('totemp', 'sum'),
    residential_units=('residential_units', 'sum'),
    non_residential_sqft=('non_residential_sqft', 'sum'),
    tothh=('tothh', 'sum'),
    hhq1=('hhq1', 'sum'),
    hhq2=('hhq2', 'sum'),
    hhq3=('hhq3', 'sum'),
    hhq4=('hhq4', 'sum')
).reset_index()
print('rows of inundation_summary:', inundation_summary.shape[0])
display(inundation_summary)

rows of inundation_summary: 2


,inundation_tag,totemp,residential_units,non_residential_sqft,tothh,hhq1,hhq2,hhq3,hhq4
0,False,5384365.0,3977679.0,2.350200e+09,3783034.0,939122.0,826855.0,739848.0,1277209.0
1,True,51254.0,13272.0,2.692608e+07,12798.0,4356.0,2888.0,1941.0,3613.0


In [7]:
# calculated inundation risk as share of total
for i in ['totemp', 'residential_units', 'non_residential_sqft', 'tothh', 'hhq1', 'hhq2', 'hhq3', 'hhq4']:
    inundation_summary[f'{i}_share'] = inundation_summary[i] / inundation_summary[i].sum()

total_row = inundation_summary.drop(columns="inundation_tag").sum(numeric_only=True)
total_row["inundation_tag"] = "Total"

inundation_summary = pd.concat(
    [inundation_summary, total_row.to_frame().T],
    ignore_index=True,
)

inundation_summary = inundation_summary[
    [inundation_summary.columns[0]] + sorted(inundation_summary.columns[1:])
]

display(inundation_summary)
inundation_summary['directory'] = 'PBA50Plus_Final_Blueprint_v65'
inundation_summary.to_csv(r'M:\urban_modeling\baus\PBA60\Strategy_evaluation\metrics\inundation_summary.csv')


,inundation_tag,hhq1,hhq1_share,hhq2,hhq2_share,hhq3,hhq3_share,hhq4,hhq4_share,non_residential_sqft,non_residential_sqft_share,residential_units,residential_units_share,totemp,totemp_share,tothh,tothh_share
0,False,939122.0,0.995383,826855.0,0.996519,739848.0,0.997383,1277209.0,0.997179,2350199531.941162,0.988673,3977679.0,0.996674,5384365.0,0.990571,3783034.0,0.996628
1,True,4356.0,0.004617,2888.0,0.003481,1941.0,0.002617,3613.0,0.002821,26926079.697589,0.011327,13272.0,0.003326,51254.0,0.009429,12798.0,0.003372
2,Total,943478.0,1.0,829743.0,1.0,741789.0,1.0,1280822.0,1.0,2377125611.638751,1.0,3990951.0,1.0,5435619.0,1.0,3795832.0,1.0


In [8]:
# just to summarize 2020 to help with debug
parcels_2020 = pd.read_csv(MODEL_DIR / "core_summaries" / "PBA50Plus_Final_Blueprint_v65_parcel_summary_2020.csv")
print('rows of parcels_2020:', parcels_2020.shape[0], '; unique parcel_id:', parcels_2020['parcel_id'].nunique())
display(parcels_2020.head(3))

# merge
parcels_withSLR_2020 = pd.merge(
    parcels_2020[['parcel_id', 'totemp', 'residential_units', 'non_residential_sqft', 'tothh', 'hhq1', 'hhq2', 'hhq3', 'hhq4']],
    slr_lookup,
    on='parcel_id',
    how='left'
)
print('rows of parcels_withSLR_2020:', parcels_withSLR_2020.shape[0], '; unique parcel_id:', parcels_withSLR_2020['parcel_id'].nunique())
display(parcels_withSLR_2020.head(3))


# summarize inundation impacts
parcels_withSLR_2020['inundation_tag'] = parcels_withSLR_2020['inundation'].notnull()
inundation_summary_2020 = parcels_withSLR_2020.groupby('inundation_tag').agg(
    totemp=('totemp', 'sum'),
    residential_units=('residential_units', 'sum'),
    non_residential_sqft=('non_residential_sqft', 'sum'),
    tothh=('tothh', 'sum'),
    hhq1=('hhq1', 'sum'),
    hhq2=('hhq2', 'sum'),
    hhq3=('hhq3', 'sum'),
    hhq4=('hhq4', 'sum')
).reset_index()
print('rows of inundation_summary_2020:', inundation_summary_2020.shape[0])
display(inundation_summary_2020)

rows of parcels_2020: 1956212 ; unique parcel_id: 1956212


,parcel_id,geom_id,y,x,max_far,built_dua,max_dua,built_far,non_residential_sqft,residential_units,...,hhq3,hhq4,tothh,AGREMPN,FPSEMPN,HEREMPN,MWTEMPN,OTHEMPN,RETEMPN,totemp
0,229116,10305106092872,37.655379,-121.795620,0.00,0.000000,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,244166,11107351665227,37.717277,-121.713004,0.35,0.000000,3.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,202378,11030175960628,37.655260,-122.014199,0.00,1.133806,0.0,0.116293,0.0,17.0,...,3.0,5.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


rows of parcels_withSLR_2020: 1956212 ; unique parcel_id: 1956212


,parcel_id,totemp,residential_units,non_residential_sqft,tothh,hhq1,hhq2,hhq3,hhq4,inundation
0,229116,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,244166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,202378,0.0,17.0,0.0,17.0,2.0,7.0,3.0,5.0,NaN


rows of inundation_summary_2020: 2


,inundation_tag,totemp,residential_units,non_residential_sqft,tothh,hhq1,hhq2,hhq3,hhq4
0,False,3997002.0,2966995.0,2.306908e+09,2785998.0,600120.0,594743.0,659355.0,931780.0
1,True,27916.0,8128.0,1.681996e+07,7525.0,1685.0,1600.0,1698.0,2542.0
